# Duality and Complements

**Part II · Geometric Algebra** — Tutorial 08

This tutorial covers the dual and complement operations in pytanga: the unsigned
bitwise **complement** (`complement()`), the signed Clifford **dual** (`dual()`),
its inverse the **undual** (`undual()`), and the **left dual** (`ldual()`). The four
operations answer the same question — *“what is the 'opposite' blade?”* — but differ
in whether and how they carry signs.

By the end you will be able to:

- Distinguish the unsigned `complement()` from the signed `dual()`.
- Compute the **Hodge dual** `★A = A · I⁻¹` and recover the vector cross product.
- Use the **left dual** `I · A` where the pseudoscalar is not invertible.
- Invert the dual with `undual()`, satisfying `dual(undual(A)) == A`.
- Predict the dual-of-dual sign across the eight basis algebras.
- Read the fixed Hodge-star conventions of `BasisPGA3` / `BasisPGA2`.
- Map between **IPNS and OPNS**, build the **regressive product** via duality,
  and extract **normals / orthogonal complements**.

> **Prerequisites:** [Tutorial 02](../02_algebra_core/) (products, grades, reverse),
> [Tutorial 03](../03_basis_classes/) (the eight basis classes), and
> [Tutorial 06](../06_conformal_n3/) (IPNS vs OPNS). Geometric entities are created
> through the `pytanga.geometry` submodule.


## 1. Setup

All four operations are methods on `MV` and also exist on the `Algebra` as
`alg.complement(a)`, `alg.dual(a)`, `alg.ldual(a)`, and `alg.undual(a)`. We import
the basis classes we need and bind a `Geometry` for the entity examples.


In [1]:
from pytanga.basis import (
    BasisE2,
    BasisE3,
    BasisN2,
    BasisN3,
    BasisP2,
    BasisP3,
    BasisPGA2,
    BasisPGA3,
)
from pytanga.geometry import Geometry, Point, Sphere

E3 = BasisE3()
geo = Geometry(E3)   # binds the algebra; OPNS/IPNS read from E3.opns (default True)


## 2. Complement — the unsigned bitwise complement

`complement()` is the **purely combinatorial** complement: each basis blade is
mapped to the blade made of *all the basis vectors it does not contain* (a bitwise
`XOR` with the pseudoscalar's blade mask). No sign is applied to the coefficients,
so the operation is an involution in every algebra:

    complement(complement(A)) == A

Because it ignores sign it is **not** the Clifford dual: it does not satisfy
geometric identities such as `★(a ∧ b) = a × b`. Use it for bitmask / index
gymnastics, blade-mask tracking, or anywhere the sign is handled separately.

> **Naming note:** in pytanga `~MV` is the **reverse** (see Tutorial 02), not the
> complement. The unsigned complement is spelled `complement()`.


In [2]:
a = E3("2 e1 + 3 e12")

a.show("a")
a.complement().show("complement(a)  (e1 ↔ e23,  e12 ↔ e3,  signs unchanged)")
print("complement(complement(a)) == a:", (a.complement().complement() - a).is_zero)
print("~a (reverse, NOT the complement) =", ~a)


a: 2 e1 + 3 e12

complement(a)  (e1 ↔ e23,  e12 ↔ e3,  signs unchanged): 3 e3 + 2 e23

complement(complement(a)) == a: True
~a (reverse, NOT the complement) = 2 e1 - 3 e12


## 3. Signed dual — dual()

The signed dual is the standard Clifford (Hodge) dual:

    ★A = A · I⁻¹

where `I⁻¹` is the inverse pseudoscalar. The blade mask is the same bitwise
complement as `complement()`, but each coefficient also receives the sign of the
permutation that reorders the blade with the pseudoscalar into canonical order.
That sign is what makes `dual()` geometrically correct: in G(3,0) it turns the
outer product of two vectors into the vector cross product:

    ★(a ∧ b) = a × b


In [3]:
# Dual of each basis blade in E3 (G(3,0), I² = −1)
for name in ["e1", "e2", "e3", "e12", "e13", "e23", "I"]:
    mv = E3(name)
    print(f"★{name:>4} = {mv.dual()}")

# The cross product is the Hodge dual of the outer product:  a × b = ★(a ∧ b)
a = E3("1 e1 + 2 e2 + 3 e3")
b = E3("4 e1 + 5 e2 + 6 e3")

(a ^ b).show("a ^ b")
(a.op(b).dual()).show("★(a ^ b)  =  a × b")


★  e1 = -e23
★  e2 = e13
★  e3 = -e12
★ e12 = e3
★ e13 = -e2
★ e23 = e1
★   I = 1


a ^ b: - 3 e12 - 6 e13 - 3 e23

★(a ^ b)  =  a × b: - 3 e1 + 6 e2 - 3 e3

## 4. Left dual — ldual()

The left dual multiplies by the pseudoscalar from the *left*:

    ldual(A) = I · A

It does **not** need the pseudoscalar inverse, which makes it the right tool in
algebras where the pseudoscalar is null (e.g. the plane-based PGA models, where
`I² = 0`). In G(3,0) the pseudoscalar commutes with everything and `I⁻¹ = −I`,
so the left dual is simply the negation of the right dual for every element.


In [4]:
for name in ["e1", "e12", "I"]:
    mv = E3(name)
    print(f"I·{name:>4} = {mv.ldual()}")

# In G(3,0):  I commutes with everything and I⁻¹ = −I,  so ldual(A) = −dual(A)
print("ldual(e1)  == -dual(e1) :", (E3("e1").ldual() + E3("e1").dual()).is_zero)
print("ldual(e12) == -dual(e12):", (E3("e12").ldual() + E3("e12").dual()).is_zero)


I·  e1 = e23
I· e12 = -e3
I·   I = -1
ldual(e1)  == -dual(e1) : True
ldual(e12) == -dual(e12): True


## 5. Undual — the inverse of the dual

`undual()` is the inverse of the signed dual: it multiplies by the pseudoscalar on
the right,

    undual(A) = A · I

so that `dual(undual(A)) == A`. In algebras with an invertible pseudoscalar
(E3, P3, N3, …) this is exactly the right inverse of `dual()`.


In [5]:
A = E3("2 e1 + 3 e12")

A.undual().show("undual(A)  = A · I")
print("dual(undual(A)) == A:", (A.undual().dual() - A).is_zero)
print("undual(dual(A)) == A:", (A.dual().undual() - A).is_zero)


undual(A)  = A · I: - 3 e3 + 2 e23

dual(undual(A)) == A: True
undual(dual(A)) == A: True


## 6. Dual-of-dual — the sign table

Applying `dual()` twice is not always the identity. For the metric algebras
(those using the standard `A · I⁻¹` dual) the sign is

    ★★A = (−1)^(D(D−1)/2 + s) · A

where `D` is the dimension and `s` the number of negative-signature basis
vectors. The table below covers the six metric basis classes; the two PGA classes
use the J-map and are treated in the next section.


In [6]:
bases = [
    ("E3", BasisE3), ("P3", BasisP3), ("N3", BasisN3),
    ("E2", BasisE2), ("P2", BasisP2), ("N2", BasisN2),
]

print(f"{'alg':>4} {'dim':>3} {'s':>2}   ★★e1")
for name, B in bases:
    alg = B()
    v = alg("e1")
    s = alg.sig.bit_count()          # number of basis vectors that square to −1
    print(f"{name:>4} {alg.dim:>3} {s:>2}   {v.dual().dual()}")


 alg dim  s   ★★e1
  E3   3  0   -e1
  P3   4  0   e1
  N3   5  1   -e1
  E2   2  0   -e1
  P2   3  0   -e1
  N2   4  1   -e1


## 7. Hodge-star conventions in PGA3 and PGA2

The plane-based algebras `BasisPGA3` / `BasisPGA2` have a **null** pseudoscalar
(`I₄² = 0` / `I₃² = 0`), so the metric dual `A · I⁻¹` does not exist there. They
override `dual()` with Gunn's **J-map** / Dorst's Hodge star, a combinatorial
complement that satisfies

    x ∧ dual(x) = +I     for every subspace blade x

(no sign on the pseudoscalar). `undual()` is overridden to match: in 4D PGA it is
`grade_involution(dual(x))` (the double Hodge dual is the grade involution), while
in 3D PGA the J-map is involutive and `undual == dual`.


In [7]:
PGA3 = BasisPGA3()
e0, e1, e2, e3 = PGA3.e0, PGA3.e1, PGA3.e2, PGA3.e3
I4 = e0 ^ e1 ^ e2 ^ e3

for name, b in [("e0", e0), ("e1", e1), ("e2", e2), ("e3", e3),
                ("e01", e0 ^ e1), ("e12", e1 ^ e2), ("I4", I4)]:
    d = b.dual()
    print(f"★{name:>4} = {d}    {name} ∧ ★{name} = {b ^ d}")

print("dual(undual(e1)) == e1:", (PGA3.dual(PGA3.undual(e1)) - e1).is_zero)
print("undual(dual(e1)) == e1:", (PGA3.undual(PGA3.dual(e1)) - e1).is_zero)


★  e0 = e123    e0 ∧ ★e0 = I
★  e1 = e032    e1 ∧ ★e1 = I
★  e2 = e013    e2 ∧ ★e2 = I
★  e3 = e021    e3 ∧ ★e3 = I
★ e01 = e23    e01 ∧ ★e01 = I
★ e12 = e03    e12 ∧ ★e12 = I
★  I4 = 1    I4 ∧ ★I4 = I
dual(undual(e1)) == e1: True
undual(dual(e1)) == e1: True


In [8]:
PGA2 = BasisPGA2()
e0, e1, e2 = PGA2.e0, PGA2.e1, PGA2.e2

print("★e1 =", e1.dual(), "   ★★e1 =", e1.dual().dual())
print("undual(e1) == dual(e1):", (PGA2.undual(e1) - PGA2.dual(e1)).is_zero)


★e1 = e20    ★★e1 = e1
undual(e1) == dual(e1): True


## 8. Use case 1 — IPNS ↔ OPNS mapping

In the conformal model the same sphere has two representations: the **OPNS**
grade-4 blade (the wedge of four conformal points on the sphere) and the **IPNS**
grade-1 vector `c − ½·r²·einf` (the dual description). They are related by the
dual:

    dual(ipns) = opns,     dual(opns) = −ipns

The extra sign is the N3 dual-of-dual sign (−1). This is the operation behind
`pytanga.geometry`'s OPNS/IPNS switch.


In [9]:
N3 = BasisN3()
geo_n3 = Geometry(N3)

N3.opns = True
s_opns = geo_n3.create(Sphere(center=Point(1, 2, 3), radius=5.0))
N3.opns = False
s_ipns = geo_n3.create(Sphere(center=Point(1, 2, 3), radius=5.0))
N3.opns = True

s_opns.show("OPNS sphere (grade-4 blade)")
s_ipns.show("IPNS sphere (grade-1 vector  c − ½·r²·einf)")

print("dual(ipns) ==  opns:", (s_ipns.dual() - s_opns).is_zero)
print("dual(opns) == -ipns:", (s_opns.dual() + s_ipns).is_zero)


OPNS sphere (grade-4 blade): - 5.5 e123∧einf - e123∧eo - 3 e12∧einf∧eo + 2 e13∧einf∧eo - e23∧einf∧eo

IPNS sphere (grade-1 vector  c − ½·r²·einf): e1 + 2 e2 + 3 e3 - 5.5 einf + eo

dual(ipns) ==  opns: True
dual(opns) == -ipns: True


## 9. Use case 2 — regressive product via duality

The **regressive product** `∨` (the “meet”) is the dual of the outer product of the
duals:

    A ∨ B = ⋆(⋆A ∧ ⋆B)

For two spheres in N3 this is their intersection circle — the same result
`N3.meet()` returns (up to a scalar factor, since blades are projective).


In [10]:
S1 = geo_n3.create(Sphere(center=Point(0, 0, 0), radius=2.0))
S2 = geo_n3.create(Sphere(center=Point(1, 0, 0), radius=1.5))

meet = N3.meet(S1, S2)                    # built-in regressive product
reg = (S1.dual() ^ S2.dual()).dual()      # ⋆(⋆S1 ∧ ⋆S2)

print("N3.meet(S1, S2)   ->", geo_n3.analyze(meet))
print("⋆(⋆S1 ∧ ⋆S2)       ->", geo_n3.analyze(reg))
print("same circle (normalized):", (meet.normalized() - reg.normalized()).is_zero)


N3.meet(S1, S2)   -> Circle(c=Point(1.37, 0.00, 0.00), r=1.45, n=Dir(1.00, -0.00, -0.00))
⋆(⋆S1 ∧ ⋆S2)       -> Circle(c=Point(1.38, 0.00, 0.00), r=1.45, n=Dir(1.00, -0.00, -0.00))
same circle (normalized): True


## 10. Use case 3 — normals and orthogonal complements

In Euclidean 3D the dual of a bivector (an oriented plane) is its **normal**
vector, and the dual of that normal returns the plane (up to the dual-of-dual
sign). This is the same Hodge duality as the cross-product identity from
Section 3, and it is how `analyze()` reads a bivector back as an oriented `Plane`.


In [11]:
B = E3("e12")           # the xy-plane (oriented area element)
n = B.dual()            # ★e12 = e3 — the plane's normal

B.show("bivector e12 (oriented plane)")
n.show("★e12 (normal / orthogonal complement)")
print("analyze(e12):", geo.analyze(B))
print("dual(★e12)  :", n.dual(), "  (back to the plane, up to sign)")


bivector e12 (oriented plane): e12

★e12 (normal / orthogonal complement): e3

analyze(e12): Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, 0.00, 1.00))
dual(★e12)  : -e12   (back to the plane, up to sign)


## 11. Visual examples

Two figures. First, a bivector (oriented plane) and its Hodge dual — the normal
vector. Second, the OPNS/IPNS duality of a sphere in N3: the grade-4 OPNS sphere
and the grade-1 IPNS sphere describe the *same* sphere. Viewer setup is covered in
[Part I — Visualization](../../visualization/).


In [12]:
from pytanga.viz import Visualizer

B = E3("2 e12")               # the xy-plane (an oriented area element)
n = B.dual()                  # its Hodge dual → the +z normal vector

viz = Visualizer(title="E3 — a bivector (oriented plane) and its Hodge dual normal")
viz.add(B, color="#44ff44", opacity=0.3, label="bivector e12 (oriented plane)")
viz.add(n, color="#ffcc00", label="★e12 = e3 (normal)")
viz.display_snapshot()


In [13]:
N3.opns = True
s_opns = geo_n3.create(Sphere(center=Point(1, 0, 0), radius=2.0))

N3.opns = False
s_ipns = geo_n3.create(Sphere(center=Point(1, 0, 0), radius=2.0))

viz2 = Visualizer(title="N3 — the same sphere in OPNS (grade 4) and IPNS (grade 1)")
N3.opns = True
viz2.new(s_opns, color="#4488ff", opacity=0.35, label="OPNS sphere")
N3.opns = False
viz2.new(s_ipns, color="#44ff44", opacity=0.35, label="IPNS sphere")
N3.opns = True
viz2.display_snapshot()


In [14]:
import os

os.makedirs("_output/08_duality", exist_ok=True)
viz.export_snapshot("_output/08_duality/bivector_normal.html", overwrite=True)
viz2.export_snapshot("_output/08_duality/sphere_ipns_opns.html", overwrite=True)
print("Exported _output/08_duality/bivector_normal.html")
print("Exported _output/08_duality/sphere_ipns_opns.html")


Exported _output/08_duality/bivector_normal.html
Exported _output/08_duality/sphere_ipns_opns.html


## 12. Summary & next steps

You now know pytanga's four complement/dual operations:

| Operation | Formula | Sign? | Use |
|---|---|---|---|
| `complement()` | bitwise `XOR` with `I` | no | bitmask / index gymnastics |
| `dual()` | `A · I⁻¹` | yes (Clifford) | Hodge dual, OPNS↔IPNS, cross product |
| `ldual()` | `I · A` | yes | when `I` is not invertible (PGA) |
| `undual()` | `A · I` | yes | inverse of `dual()` |

| Concept | API |
|---|---|
| Unsigned complement | `a.complement()` |
| Hodge dual / cross product | `a.dual()`, `a.op(b).dual()` |
| Left dual | `a.ldual()` |
| Inverse dual | `a.undual()` |
| IPNS ↔ OPNS | `sphere.dual()` (N3) |
| Regressive product | `A.dual().op(B.dual()).dual()` |
| Plane normal | `bivector.dual()` |

**Where to go next:**

- [**09 · Modulus Arithmetic with Integer Algebras**](../09_modulus/) — the next
  tutorial in the series.
- [**06 · Conformal 3D**](../06_conformal_n3/) — the IPNS/OPNS duality in context.
